# GLES MIP Pilot Sampling

Create a reproducible pilot sample from `pre020s` only: the pre-election open-ended response to the question, "What is the most important political problem facing Germany today?"

This sample is designed for structured coding and inter-model agreement analysis, not population representativeness.

## 1. Setup

Use a fixed seed for reproducibility. The raw files stay unchanged; the derived sample is written to `data/stance_ambiguity/interim/`.

In [1]:
from pathlib import Path
import re

import pandas as pd

SAMPLE_SEED = 2025
SOURCE_VAR = "pre020s"

PROJECT_ROOT = Path.cwd().parents[1]
RAW_DIR = PROJECT_ROOT / "data" / "stance_ambiguity" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "stance_ambiguity" / "interim"

open_ended_path = RAW_DIR / "ZA10101_v3-0-0_open-ended.csv"
sav_path = RAW_DIR / "ZA10101_v3-0-0_en.sav"
sample_output_path = INTERIM_DIR / "gles_mip_sample.csv"

print("Open-ended CSV:", open_ended_path)
print("SAV metadata:", sav_path)
print("Sample output:", sample_output_path)

Open-ended CSV: /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/raw/ZA10101_v3-0-0_open-ended.csv
SAV metadata: /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/raw/ZA10101_v3-0-0_en.sav
Sample output: /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/interim/gles_mip_sample.csv


## 2. Load Raw Data

The open-ended CSV is semicolon-delimited and uses a Latin encoding. We read values as strings so response text is preserved.

In [2]:
open_ended = pd.read_csv(open_ended_path, sep=";", encoding="latin1", dtype=str)
sav = pd.read_spss(sav_path)

print("Open-ended shape:", open_ended.shape)
print("SAV shape:", sav.shape)
print("pre020s present:", SOURCE_VAR in open_ended.columns)

Open-ended shape: (8562, 21)
SAV shape: (8562, 803)
pre020s present: True


## 3. Prepare IDs and Minimal Text Cleaning

Filtering removes structural missing/status codes and clearly non-codable placeholder responses. Cleaning is intentionally minimal: trim whitespace, collapse repeated internal whitespace, and preserve substantive wording.

In [4]:
def normalize_lfdn(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype(str)


def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text


missing_tokens = {
    "",
    "-97 trifft nicht zu",
    "-95 nicht teilgenommen",
    "-93 Interview abgebrochen",
    "-98 weiss nicht",
    "-98 weiß nicht",
    "-99 keine Angabe",
}
missing_tokens_folded = {token.casefold() for token in missing_tokens}


def is_non_codable(text):
    if pd.isna(text):
        return True
    folded = str(text).casefold()
    if folded in missing_tokens_folded:
        return True
    if re.fullmatch(r"[\W_]+", str(text), flags=re.UNICODE):
        return True
    if len(str(text)) == 1 and str(text).casefold() in {"x", "y"}:
        return True
    return False


def count_words(text):
    return len(re.findall(r"\w+", str(text), flags=re.UNICODE))

## 4. Build the `pre020s` Sampling Frame

The sampling unit is one response row from `pre020s`. We keep both raw and minimally cleaned response text.

In [5]:
open_ended["lfdn_key"] = normalize_lfdn(open_ended["lfdn"])

frame = open_ended[["lfdn", "lfdn_key", SOURCE_VAR]].copy()
frame = frame.rename(columns={SOURCE_VAR: "response_text_raw"})
frame["source_var"] = SOURCE_VAR
frame["response_text"] = frame["response_text_raw"].map(clean_text)
frame["is_non_codable"] = frame["response_text"].map(is_non_codable)

filtered = frame.loc[~frame["is_non_codable"]].copy()
filtered["n_words"] = filtered["response_text"].map(count_words)
filtered = filtered.loc[filtered["n_words"] > 0].copy()
filtered["n_chars"] = filtered["response_text"].str.len()
filtered["response_norm"] = filtered["response_text"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
filtered["response_freq"] = filtered.groupby("response_norm")["response_norm"].transform("size")

print("Raw pre020s rows:", len(frame))
print("Filtered codable pre020s rows:", len(filtered))
print("Excluded rows:", len(frame) - len(filtered))

filtered.head()

Raw pre020s rows: 8562
Filtered codable pre020s rows: 8316
Excluded rows: 246


,lfdn,lfdn_key,response_text_raw,source_var,response_text,is_non_codable,n_words,n_chars,response_norm,response_freq
0,91,91,Migration,pre020s,Migration,False,1,9,migration,1128
1,92,92,migranten,pre020s,migranten,False,1,9,migranten,44
2,93,93,Wirtschaftliche Situation,pre020s,Wirtschaftliche Situation,False,2,25,wirtschaftliche situation,22
3,94,94,Einigkeit,pre020s,Einigkeit,False,1,9,einigkeit,14
4,95,95,Sozialbetrug,pre020s,Sozialbetrug,False,1,12,sozialbetrug,1


## 5. Create Word-Count Strata

Word count is based on the minimally cleaned response text. Strata are fixed before sampling.

In [8]:
def assign_stratum(n_words):
    if n_words == 1:
        return "single_word"
    if 2 <= n_words <= 4:
        return "short_phrase"
    if n_words >= 5:
        return "elaborated"
    return pd.NA


filtered["stratum"] = filtered["n_words"].map(assign_stratum)

stratum_targets = {
    "single_word": 120,
    "short_phrase": 120,
    "elaborated": 160,
}

stratum_counts = filtered["stratum"].value_counts().reindex(stratum_targets.keys(), fill_value=0)
stratum_counts


stratum
single_word     5639
short_phrase    1957
elaborated       720
Name: count, dtype: int64

## 6. Stratified Sampling

Sampling is row-level, not unique-text-level. If a stratum has fewer available rows than requested, the notebook samples all available rows and prints a warning.

In [9]:
sample_parts = []

for stratum, target_n in stratum_targets.items():
    stratum_frame = filtered.loc[filtered["stratum"] == stratum]
    available_n = len(stratum_frame)
    sample_n = min(target_n, available_n)

    if available_n < target_n:
        print(f"Warning: {stratum} has only {available_n} available rows; requested {target_n}. Sampling all available rows.")

    sample_parts.append(stratum_frame.sample(n=sample_n, random_state=SAMPLE_SEED))

sample = pd.concat(sample_parts, ignore_index=True)
sample = sample.sample(frac=1, random_state=SAMPLE_SEED).reset_index(drop=True)
sample.insert(0, "sample_id", [f"gles_mip_v1_{i:04d}" for i in range(1, len(sample) + 1)])
sample["sample_seed"] = SAMPLE_SEED

print("Sample size:", len(sample))
display(sample["stratum"].value_counts().reindex(stratum_targets.keys(), fill_value=0))

sample.head()

Sample size: 400


stratum
single_word     120
short_phrase    120
elaborated      160
Name: count, dtype: int64

,sample_id,lfdn,lfdn_key,response_text_raw,source_var,response_text,is_non_codable,n_words,n_chars,response_norm,response_freq,stratum,sample_seed
0,gles_mip_v1_0001,9122,9122,Wirtschaft,pre020s,Wirtschaft,False,1,10,wirtschaft,1133,single_word,2025
1,gles_mip_v1_0002,8513,8513,Wirtschaft,pre020s,Wirtschaft,False,1,10,wirtschaft,1133,single_word,2025
2,gles_mip_v1_0003,3755,3755,Migration,pre020s,Migration,False,1,9,migration,1128,single_word,2025
3,gles_mip_v1_0004,6368,6368,Asylpolitik,pre020s,Asylpolitik,False,1,11,asylpolitik,143,single_word,2025
4,gles_mip_v1_0005,92,92,migranten,pre020s,migranten,False,1,9,migranten,44,single_word,2025


## 7. Add Metadata

Carry a small set of respondent metadata from the `.sav` file when available.

In [11]:
sav["lfdn_key"] = normalize_lfdn(sav["lfdn"])

metadata_cols = ["lfdn_key", "pre001", "ostwest", "sup057"]
available_metadata_cols = [col for col in metadata_cols if col in sav.columns]
missing_metadata_cols = [col for col in metadata_cols if col not in sav.columns]

print("Available metadata:", available_metadata_cols)
print("Missing metadata:", missing_metadata_cols)

metadata = sav[available_metadata_cols].copy()
sample = sample.merge(metadata, on="lfdn_key", how="left", validate="many_to_one")

sample.head()

Available metadata: ['lfdn_key', 'pre001', 'ostwest', 'sup057']
Missing metadata: []


,sample_id,lfdn,lfdn_key,response_text_raw,source_var,response_text,is_non_codable,n_words,n_chars,response_norm,response_freq,stratum,sample_seed,pre001,ostwest,sup057
0,gles_mip_v1_0001,9122,9122,Wirtschaft,pre020s,Wirtschaft,False,1,10,wirtschaft,1133,single_word,2025,female,West Germany,7
1,gles_mip_v1_0002,8513,8513,Wirtschaft,pre020s,Wirtschaft,False,1,10,wirtschaft,1133,single_word,2025,male,West Germany,6
2,gles_mip_v1_0003,3755,3755,Migration,pre020s,Migration,False,1,9,migration,1128,single_word,2025,male,West Germany,4
3,gles_mip_v1_0004,6368,6368,Asylpolitik,pre020s,Asylpolitik,False,1,11,asylpolitik,143,single_word,2025,male,Unit nonresponse,Unit nonresponse
4,gles_mip_v1_0005,92,92,migranten,pre020s,migranten,False,1,9,migranten,44,single_word,2025,male,West Germany,9


## 8. Write Sample CSV

The output file is a derived interim dataset for pilot coding. Raw data are not modified.

In [14]:
required_output_cols = [
    "sample_id",
    "lfdn",
    "lfdn_key",
    "source_var",
    "response_text_raw",
    "response_text",
    "n_words",
    "n_chars",
    "stratum",
    "response_freq",
    "sample_seed",
]
metadata_output_cols = [col for col in ["pre001", "ostwest", "sup057"] if col in sample.columns]
output_cols = required_output_cols + metadata_output_cols

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
sample_output = sample[output_cols].copy()
sample_output.to_csv(sample_output_path, index=False)

print("Wrote:", sample_output_path)
print("Rows:", len(sample_output))
display(sample_output["stratum"].value_counts().reindex(stratum_targets.keys(), fill_value=0))

sample_output.head()

Wrote: /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/interim/gles_mip_sample.csv
Rows: 400


stratum
single_word     120
short_phrase    120
elaborated      160
Name: count, dtype: int64

,sample_id,lfdn,lfdn_key,source_var,response_text_raw,response_text,n_words,n_chars,stratum,response_freq,sample_seed,pre001,ostwest,sup057
0,gles_mip_v1_0001,9122,9122,pre020s,Wirtschaft,Wirtschaft,1,10,single_word,1133,2025,female,West Germany,7
1,gles_mip_v1_0002,8513,8513,pre020s,Wirtschaft,Wirtschaft,1,10,single_word,1133,2025,male,West Germany,6
2,gles_mip_v1_0003,3755,3755,pre020s,Migration,Migration,1,9,single_word,1128,2025,male,West Germany,4
3,gles_mip_v1_0004,6368,6368,pre020s,Asylpolitik,Asylpolitik,1,11,single_word,143,2025,male,Unit nonresponse,Unit nonresponse
4,gles_mip_v1_0005,92,92,pre020s,migranten,migranten,1,9,single_word,44,2025,male,West Germany,9
